# Bird Identifier: training on Kaggle

Fine-tunes `convnext_nano` on 300 European bird species at 288 px and exports it to ONNX for the browser.

**Before running:**
1. Upload all the `ml/dist/bird-identifier-data-*.zip` parts to **one** Kaggle Dataset (*Datasets → New Dataset*, add every part), then attach it to this notebook (*Add Input*).
2. *Settings → Accelerator*: **GPU T4 x2** (training uses both GPUs).
3. *Settings → Internet*: **on** (needed to download the pretrained weights; requires a phone-verified Kaggle account).
4. Click **Save Version → Save & Run All** to train in the background (roughly 3–5 hours), or run cells one by one.
   Training stops early if it would run past 10 hours, so results are always saved within Kaggle's 12-hour limit.

**Updated code without re-uploading the photos:** add `ml/dist/bird-identifier-code.zip` to the dataset (or as its own dataset); its `code/src/` is used instead of the copy in part 1.

**Two GPUs:** training runs one process per GPU (DistributedDataParallel, started with `torchrun`). The earlier `nn.DataParallel` setup leaked about 1.6 MB of RAM per step; the optional leak-probe cell below is how that was found. As a safety net, if an epoch still ends above 16 GB the training cell restarts from the last checkpoint (it prints `--- restarting … ---`), in the same run folder; nothing is lost.

**What to download:** `bird-model.zip` has everything needed afterwards: the ONNX models, labels, config, metrics and the `best.pt` they were made from.

**If a run stops early** (over 24 GB of RAM or the 10-hour limit), it still finishes the notebook and saves `runs/<run>/last.pt`. Attach that version's output to the notebook (*Add Input → Your Work → this notebook*) and run again; it resumes from the last finished epoch.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Find the dataset and set up the code

In [ ]:
import glob, os, shutil
from pathlib import Path

import re

matches = glob.glob("/kaggle/input/**/splits/train.csv", recursive=True)
assert matches, "Dataset not found: attach the uploaded dataset via 'Add Input'"
# Parts added later (e.g. data-5 with new species) carry newer splits: use the highest part
def part_number(path):
    found = re.findall(r"(\d+)", Path(path).parent.parent.name)
    return int(found[-1]) if found else 0
DATA_DIR = Path(max(matches, key=part_number)).parent.parent
print("Data:", DATA_DIR)

# Photos are spread over several parts; each part has its own raw/images/ folder
IMAGE_ROOTS = sorted({Path(p).parent for p in glob.glob("/kaggle/input/**/raw/images", recursive=True)}
                     - {DATA_DIR / "raw"})
print("Extra image folders:", *IMAGE_ROOTS, sep="\n  ")
EXTRA = " ".join(f'"{p}"' for p in IMAGE_ROOTS)

# Copy the training code to a writable folder so `python -m src.train` works.
# A newer copy uploaded as bird-identifier-code.zip (code/src/...) wins over the one in the data.
code = glob.glob("/kaggle/input/**/code/src/train.py", recursive=True)
SRC = Path(code[0]).parent if code else DATA_DIR / "src"
print("Code:", SRC)
WORK = Path("/kaggle/working/ml")
shutil.copytree(SRC, WORK / "src", dirs_exist_ok=True)

# If the output of an interrupted run is attached as input, carry on from it
last = sorted(glob.glob("/kaggle/input/**/runs/*/last.pt", recursive=True))
RESUME = f'--resume "{Path(last[-1]).parent}"' if last else ""
print("Resume:", RESUME or "no, starting fresh")
os.chdir(WORK)

!pip install -q onnx onnxscript onnxruntime

## Optional: measure the memory leak (instead of training)

Set `LEAK_PROBE = True` and run this cell after the setup cell (about 15 minutes on GPU T4 x2).
It trains in short bursts with one ingredient switched off at a time (DataParallel, mixed
precision, channels_last, the DataLoader) and prints how many MB the process gains per step.
Whichever change brings the growth near zero is the cause. Copy the output, then set it back
to `False` to train normally.

In [ ]:
LEAK_PROBE = False

if LEAK_PROBE:
    !python -m src.leak_probe --data-dir {DATA_DIR} --extra-image-roots {EXTRA} --img-size 288 --batch-size 64 --workers 4 --steps 300 --warmup 50 --tracemalloc
    raise SystemExit("Leak probe finished: copy the output above, then set LEAK_PROBE = False to train.")

## 2. Train

Each epoch prints training loss and validation accuracy (top-1 = the first guess is right,
top-5 = the right species is among the top 5 guesses). The best epoch is saved as `best.pt`.

In [ ]:
import time
import torch

# One training process per GPU (DistributedDataParallel via torchrun). The old
# nn.DataParallel setup leaked ~1.6 MB of RAM per step (see src/leak_probe.py).
NGPU = max(1, torch.cuda.device_count())
LAUNCH = f"torchrun --standalone --nproc_per_node={NGPU}" if NGPU > 1 else "python"

# Safety net, kept from the DataParallel days: if memory still ends an epoch above 16 GB,
# training saves, leaves a RESTART file in its run folder and exits, and a fresh process
# carries on from last.pt in the same folder. The 10-hour budget covers all restarts.
START = time.time()
resume = RESUME
for attempt in range(20):
    hours_left = 10 - (time.time() - START) / 3600
    # MALLOC_ARENA_MAX=2 stops glibc from slowly piling up memory across threads
    get_ipython().system(
        f"MALLOC_ARENA_MAX=2 {LAUNCH} -m src.train --data-dir {DATA_DIR} --extra-image-roots {EXTRA} "
        f"--out-dir /kaggle/working/runs --img-size 288 --epochs 25 --batch-size 64 --num-workers 2 "
        f"--max-hours {hours_left:.2f} --max-ram-gb 24 --restart-above-gb 16 {resume}")
    exit_code = get_ipython().user_ns["_exit_code"]
    latest = max(Path("/kaggle/working/runs").glob("*/last.pt"), key=lambda p: p.stat().st_mtime)
    if exit_code != 3 and not (latest.parent / "RESTART").exists():
        break
    resume = f'--resume "{latest.parent}"'  # carries on in the same folder
    print(f"--- restarting from {latest.parent} in a fresh process ---", flush=True)

# The folder this session trained in: best.pt, last.pt, history.csv and metrics.json
RUN = latest.parent
print("Run folder:", RUN)

## 3. Results

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

hist = pd.read_csv(RUN / "history.csv")
metrics = json.loads((RUN / "metrics.json").read_text())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist.epoch, hist.train_loss, label="train"); ax1.plot(hist.epoch, hist.val_loss, label="val")
ax1.set(title="Loss", xlabel="epoch"); ax1.legend()
ax2.plot(hist.epoch, hist.val_top1, label="top-1"); ax2.plot(hist.epoch, hist.val_top5, label="top-5")
ax2.set(title="Validation accuracy", xlabel="epoch", ylim=(0, 1)); ax2.legend()
plt.show()

print(f"Test top-1: {metrics['test_top1']:.1%}   top-5: {metrics['test_top5']:.1%}   (best epoch {metrics['best_epoch']})")
display(pd.DataFrame(metrics["per_class"]).head(10).style.set_caption("Hardest species"))
display(pd.DataFrame(metrics["most_confused"]).head(10).style.set_caption("Most confused pairs"))

## 4. Export to ONNX for the browser

In [ ]:
import torch

# The checkpoint must be the one the test results came from
best_epoch = torch.load(RUN / "best.pt", map_location="cpu", weights_only=False)["epoch"]
assert best_epoch == metrics["best_epoch"], (best_epoch, metrics["best_epoch"])

!python -m src.export_onnx {RUN}/best.pt --out /kaggle/working/export --int8
# best.pt goes in the same zip as the model made from it, so later work (fine-tuning,
# new heads) starts from exactly the weights that are deployed
for name in ["metrics.json", "history.csv", "best.pt"]:
    shutil.copy(RUN / name, "/kaggle/working/export/")
shutil.make_archive("/kaggle/working/bird-model", "zip", "/kaggle/working/export")
print(f"Download bird-model.zip from the Output panel: model.onnx, model.int8.onnx, labels.json, "
      f"config.json, metrics and best.pt (epoch {best_epoch}).")